# SCENTIA — Análisis Exploratorio de Datos (EDA) & Auditoría de Perfumería
**Proyecto Integrador | Sistema de Recomendación e Inteligencia Olfativa**  
**Autora:** Josué Jiménez Apodaca  
**Fecha:** Julio 2026  

---

## 📋 Resumen Ejecutivo & Objetivos del EDA

El presente *Exploratory Data Analysis* (EDA) tiene como propósito auditar, limpiar y caracterizar la estructura interna del catálogo masivo de fragancias extraídas mediante web scraping desde **Fragrantica**. Los hallazgos de este cuaderno constituirán el fundamento cuantitativo para el diseño del **Pipeline de Ingeniería de Variables** (*Feature Engineering*) y el desarrollo de los algoritmos de recomendación vectorial (TF-IDF + Cosine Similarity / Embeddings de Perfumería).

### Ejes Principales de Investigación:
1. **Calidad e Integridad de Datos:** Evaluación de completitud (*missingness*), tipos de datos y distribuciones nulas por campo.
2. **Análisis Univariado:**
   * Evaluación de puntuaciones globales (*Global Rating*) y volumen de votos (*Rating Count*).
   * Distribución de casas de diseño (*Designers / Brands*) y perfumistas (*Noses*).
3. **Análisis de Notas & Pirámides Olfativas:**
   * Frecuencia e insumos dominantes en notas de **Salida** (*Top Notes*), **Corazón** (*Heart Notes*) y **Fondo** (*Base Notes*).
4. **Análisis Multidimensional & Metadatos del Consumidor:**
   * Desglose de distribuciones JSON sobre uso estacional (*Seasons*), horario (*Time of Day*), longevidad (*Longevity*), estela (*Sillage*) y precio (*Price/Value*).
5. **Directrices Estratégicas para la Fase de ML:** Recomendaciones concretas de preprocesamiento, normalización e imputación.


In [ ]:
import os
import sys
import json
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética profesional para visualizaciones
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
warnings.filterwarnings('ignore')

# Paleta de colores Luxury / Fragrance (Amber, Gold, Obsidian, Emerald)
AMBER_PRIMARY = '#d97706'
AMBER_LIGHT = '#f59e0b'
SLATE_DARK = '#1e293b'
SLATE_LIGHT = '#64748b'
EMERALD_ACCENT = '#059669'
BURGUNDY_ACCENT = '#9f1239'

PALETTE = [AMBER_PRIMARY, AMBER_LIGHT, EMERALD_ACCENT, SLATE_DARK, BURGUNDY_ACCENT, SLATE_LIGHT]

print('✅ Entorno de Análisis Exploratorio cargado exitosamente.')


---
## 1. Carga de Datos y Auditoría de Completitud
Cargamos el dataset extraído/sintético y analizamos la tasa de completitud por columna.


In [ ]:
# Carga de datos con fallback resiliente
csv_possible_paths = [
    '../data/fragrantica_data_from_scraper.csv'
]

df = None
for path in csv_possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path, sep='|')
        print(f'📁 Dataset cargado desde: {path}')
        break

if df is None:
    print('⚠️ Generando dataset sintético de respaldo con la misma estructura para exploración...')
    np.random.seed(42)
    data_demo = {
        'url': [f'https://www.fragrantica.es/perfume/Brand/Perfume-{i}.html' for i in range(1, 151)],
        'bottle_image_url': ['https://images.unsplash.com/photo-1592945403244'] * 150,
        'name_raw': [f'Fragancia {i}' for i in range(1, 151)],
        'designer_raw': np.random.choice(['Dior', 'Chanel', 'Tom Ford', 'Creed', 'Versace', 'Xerjoff', 'Parfums de Marly', 'Jo Malone'], 150),
        'global_rating': np.round(np.random.normal(4.15, 0.35, 150).clip(2.0, 5.0), 2),
        'global_rating_count': np.random.randint(10, 5000, 150),
        'top_notes_raw': np.random.choice(['bergamot, lemon, pink pepper', 'lavender, mint, cardamom', 'mandarin orange, apple', None], 150),
        'heart_notes_raw': np.random.choice(['rose, jasmine, iris', 'cinnamon, rum, cedar', 'sage, geranium', None], 150),
        'base_notes_raw': np.random.choice(['vanilla, tonka bean, amber', 'sandalwood, musk, vetiver', 'oud, leather, patchouli', None], 150),
        'perfumers_raw': np.random.choice(['Olivier Polge', 'Dominique Ropion', 'Quentin Bisch', 'François Demachy', None], 150),
        'seasons_raw_dist': ['{"invierno": "35%", "otoño": "30%", "primavera": "20%", "verano": "15%"}'] * 150,
        'time_of_day_raw_dist': ['{"noche": "60%", "dia": "40%"}'] * 150,
        'longevity_raw_dist': ['{"duradera": "50%", "moderada": "30%", "muy duradera": "20%"}'] * 150,
        'sillage_raw_dist': ['{"pesada": "40%", "moderada": "45%", "suave": "15%"}'] * 150
    }
    df = pd.DataFrame(data_demo)

print(f'📊 Dimensiones del Dataset: {df.shape[0]} filas × {df.shape[1]} columnas')
display(df.head(10))


Cuando se diseño el scraper existian elementos que estaban en el mismo contenedor y los diccionarios guardaron mas de una variable, esto ocurre con longevity + sillage y price_value + gender. En el siguiente bloque trataremos de limpiar estas columnas para separar en dos cada diccionario.

In [ ]:
import json
import ast
import pandas as pd

# 1. Definición de conjuntos de llaves por cada variable
LONGEVITY_KEYS = {"escasa", "débil", "dÉbil", "d\u00e9bil", "duradera", "muy duradera"} 
# Nota: "moderada" se manejará contextualmente o se compartirá si aplica, 
# pero la incluimos en las búsquedas.

SILLAGE_KEYS = {"suave", "pesada", "enorme"}

GENDER_KEYS = {"femenino", "unisex femenino", "unisex", "unisex masculino", "masculino"}

PRICE_KEYS = {
    "extremadamente costoso", 
    "ligeramente costoso", 
    "precio moderado", 
    "buen precio", 
    "excelente precio"
}

def parse_dict(val):
    """Convierte cadenas JSON o diccionarios string a dicts de Python de forma segura."""
    if pd.isna(val) or val is None:
        return {}
    if isinstance(val, dict):
        return val
    try:
        return json.loads(val)
    except (json.JSONDecodeError, TypeError):
        try:
            return ast.literal_eval(val)
        except Exception:
            return {}

def separate_distributions(row):
    """
    Separa las distribuciones combinadas en sus campos correspondientes.
    """
    # Parsing de los campos crudos concatenados
    long_sill_dict = parse_dict(row.get('longevity_raw_dist') or row.get('sillage_raw_dist'))
    gen_price_dict = parse_dict(row.get('price_value_raw_dist') or row.get('gender_voted_raw_dist'))
    
    # 1. Separar Longevidad vs Estela
    longevity_clean = {}
    sillage_clean = {}
    
    for k, v in long_sill_dict.items():
        k_lower = k.lower().strip()
        
        if k_lower in LONGEVITY_KEYS:
            longevity_clean[k] = v
        elif k_lower in SILLAGE_KEYS:
            sillage_clean[k] = v
        elif k_lower == "moderada":
            # "moderada" es compartida por ambas métricas en la interfaz
            longevity_clean[k] = v
            sillage_clean[k] = v

    # 2. Separar Género vs Precio
    gender_clean = {}
    price_clean = {}
    
    for k, v in gen_price_dict.items():
        k_lower = k.lower().strip()
        
        if k_lower in GENDER_KEYS:
            gender_clean[k] = v
        elif k_lower in PRICE_KEYS:
            price_clean[k] = v

    return pd.Series({
        'longevity_fixed_dist': json.dumps(longevity_clean, ensure_ascii=False),
        'sillage_fixed_dist': json.dumps(sillage_clean, ensure_ascii=False),
        'gender_fixed_dist': json.dumps(gender_clean, ensure_ascii=False),
        'price_value_fixed_dist': json.dumps(price_clean, ensure_ascii=False)
    })

def process_and_save_fixed_data(df, output_path="data_fixed.csv"):
    """
    Aplica la separación de distribuciones y guarda el dataframe resultante.
    """
    fixed_cols = df.apply(separate_distributions, axis=1)
    
    # Reemplazar o añadir las columnas corregidas al DataFrame
    df_fixed = df.copy()
    for col in fixed_cols.columns:
        df_fixed[col] = fixed_cols[col]
        
    # Guardar archivo auxiliar procesado
    df_fixed.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"✅ Datos corregidos y guardados exitosamente en: {output_path}")
    
    return df_fixed


df_fixed = process_and_save_fixed_data(df, output_path="data_fixed.csv")
columns_to_drop = ['longevity_raw_dist', 'sillage_raw_dist', 'gender_voted_raw_dist', 'price_value_raw_dist']
df_fixed.drop(columns=columns_to_drop, inplace=True)

In [ ]:
# El dataset df_fixed no contiene un perfume_id explicito, lo crearemos como alfanumérico incremental para cada perfume único basado en su URL.
df_fixed['perfume_id'] = df_fixed.groupby('url').ngroup() 



In [ ]:
print(f'📊 Dimensiones del Dataset corregido: {df_fixed.shape[0]} filas × {df_fixed.shape[1]} columnas')
display(df_fixed.head(10))

### Auditoría de Valores Faltantes por Atributo


In [ ]:
missing_df = pd.DataFrame({
    'Columna': df_fixed.columns,
    'Valores Nulos': df_fixed.isnull().sum(),
    'Porcentaje Nulo (%)': (df_fixed.isnull().sum() / len(df_fixed) * 100).round(2)
}).sort_values(by='Porcentaje Nulo (%)', ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=missing_df, x='Porcentaje Nulo (%)', y='Columna', palette='YlOrBr_r')
plt.title('Auditoría de Valores Faltantes por Campo (%)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Porcentaje Nulo (%)', fontsize=11)
plt.ylabel('Atributo', fontsize=11)
for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(f'{width:.1f}%', (width + 0.5, p.get_y() + p.get_height()/2),
                    ha='left', va='center', fontsize=9, color=SLATE_DARK, fontweight='bold')
plt.tight_layout()
plt.show()


Observando los valores ausentes vemos que hay columnas que vienen completamente vacias debido a un problema que no habia podido observar durante el proceso de scraping, más tarde buscaré corregirlos, por el momento intentamos inyectar datos sinteticos en las variables: seasons, time of the day, perfumers, vibe_reactions. Dropearemos la columna accords_query_string. El campo reviews aunque hay un gran porcentaje de valores faltantes no podemos dropearla ya que es entendible que muchos perfumes tengan reseñas nulas especialmente si el perfume es muy caro o muy desconocido o nicho. Para las columnas con notas faltantes tampoco las eliminaremos pero agregaremos una etiqueta para indicar que no tienen notas declaradas 

In [ ]:
import json
import random
import numpy as np
import pandas as pd

# Lista de perfumistas sintéticos para poblar perfumers_raw
PERFUMERS_POOL = [
    "Olivier Polge", "Dominique Ropion", "Francis Kurkdjian",
    "Quentin Bisch", "Alberto Morillas", "Christine Nagel",
    "Jacques Cavallier", "Jean-Claude Ellena", "Thierry Wasser",
    "Annick Menardo", "Bertrand Duchaufour", "Sonia Constant"
]

def format_count(val: int) -> str:
    """Convierte un entero a notación con sufijo 'k' si excede los 1,000."""
    if val >= 1000:
        formatted = f"{val / 1000:.1f}k"
        return formatted.replace(".0k", "k")
    return str(val)


def generate_synthetic_perfume_metrics():
    # 1. Perfil olfativo aleatorio para coherencia sintética
    profiles = ["fresh_citrus", "heavy_oriental", "mass_pleaser", "niche_polarizing"]
    profile = random.choice(profiles)

    # 2. Generación de Ratings / Vibe Reactions (love, like, ok, dislike, hate)
    if profile == "mass_pleaser":
        base_votes = random.randint(5000, 30000)
        weights_rating = [0.50, 0.30, 0.12, 0.05, 0.03]  # love, like, ok, dislike, hate
    elif profile == "niche_polarizing":
        base_votes = random.randint(800, 8000)
        weights_rating = [0.35, 0.10, 0.05, 0.20, 0.30]
    else:
        base_votes = random.randint(10, 5000)
        weights_rating = [0.25, 0.35, 0.20, 0.12, 0.08]

    counts_rating = np.random.multinomial(base_votes, weights_rating)

    vibe_dict = {
        "love": format_count(counts_rating[0]),
        "like": format_count(counts_rating[1]),
        "ok": format_count(counts_rating[2]),
        "dislike": format_count(counts_rating[3]),
        "hate": format_count(counts_rating[4]),
    }

    # 3. Generación de Seasons & Time of Day
    if profile == "fresh_citrus":
        season_weights = [0.05, 0.35, 0.45, 0.15]  # winter, spring, summer, fall
        tod_weights = [0.75, 0.25]                 # day, night
    elif profile == "heavy_oriental":
        season_weights = [0.50, 0.10, 0.05, 0.35]
        tod_weights = [0.20, 0.80]
    else:
        season_weights = [0.25, 0.25, 0.25, 0.25]
        tod_weights = [0.50, 0.50]

    season_votes = random.randint(int(base_votes * 0.6), int(base_votes * 1.2) + 1)
    counts_season = np.random.multinomial(season_votes, season_weights)

    tod_votes = random.randint(int(base_votes * 0.5), int(base_votes * 1.1) + 1)
    counts_tod = np.random.multinomial(tod_votes, tod_weights)

    seasons_dict = {
        "winter": format_count(counts_season[0]),
        "spring": format_count(counts_season[1]),
        "summer": format_count(counts_season[2]),
        "fall": format_count(counts_season[3]),
    }

    day_time_dict = {
        "day": format_count(counts_tod[0]),
        "night": format_count(counts_tod[1]),
    }

    # 4. Generación de Perfumistas (perfumers_raw)
    # 85% probabilidad de tener 1 o 2 perfumistas, 15% de ser None / desconocido
    if random.random() < 0.85:
        num_perfumers = random.choice([1, 1, 2])
        selected_perfumers = random.sample(PERFUMERS_POOL, num_perfumers)
        perfumers_raw = ", ".join(selected_perfumers)
    else:
        perfumers_raw = None

    # Devuelve las 4 columnas formateadas como String JSON
    return {
        "vibe_reactions_raw_dist": json.dumps(vibe_dict),
        "seasons_raw_dist": json.dumps(seasons_dict),
        "time_of_day_raw_dist": json.dumps(day_time_dict),
        "perfumers_raw": perfumers_raw
    }


In [ ]:


# Generar datos sintéticos para todas las filas de df_fixed
synthetic_rows = [generate_synthetic_perfume_metrics() for _ in range(len(df_fixed))]
df_synthetic = pd.DataFrame(synthetic_rows)

# Asignar/Actualizar las columnas en df_fixed
df_fixed[["vibe_reactions_raw_dist", "seasons_raw_dist", "time_of_day_raw_dist", "perfumers_raw"]] = df_synthetic[
    ["vibe_reactions_raw_dist", "seasons_raw_dist", "time_of_day_raw_dist", "perfumers_raw"]
]
# 2. Dropear columna inútil
df_fixed.drop(columns=['accords_query_string'], inplace=True, errors='ignore')

# 3. Tratar notas faltantes agregando la etiqueta explícita
note_cols = ['top_notes_raw', 'heart_notes_raw', 'base_notes_raw']
for col in note_cols:
    df_fixed[col] = df_fixed[col].fillna('Sin notas declaradas')

# 4. Limpieza complementaria para el resto de nulos (opcional/recomendado)
# 'reviews_text_corpus' se conserva intacta con sus NaNs/cadenas vacías
# Para ratings con 1.2% nulo, podemos rellenar con valores nulos explícitos o 0
df_fixed['global_rating'] = df_fixed['global_rating'].fillna(0.0)
df_fixed['global_rating_count'] = df_fixed['global_rating_count'].fillna(0)

In [ ]:
print(f'📊 Dimensiones del Dataset final corregido: {df_fixed.shape[0]} filas × {df_fixed.shape[1]} columnas')
print(f'head del dataset final corregido:')
df_fixed.head()

---
## 2. Análisis Univariado: Ratings, Marcas y Diseñadores


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Distribución del Rating Global
sns.histplot(df['global_rating'].dropna(), kde=True, color=AMBER_PRIMARY, ax=axes[0], bins=20)
axes[0].set_title('Distribución de Calificación Global (Global Rating)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Calificación (0 a 5 estrellas)')
axes[0].set_ylabel('Frecuencia')
mean_rating = df['global_rating'].mean()
axes[0].axvline(mean_rating, color=BURGUNDY_ACCENT, linestyle='--', label=f'Media: {mean_rating:.2f}')
axes[0].legend()

# 2. Relación Rating vs Número de Votos
sns.scatterplot(data=df, x='global_rating_count', y='global_rating', color=EMERALD_ACCENT, alpha=0.7, ax=axes[1])
axes[1].set_title('Rating Global vs. Volumen de Evaluaciones', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cantidad de Votos (Rating Count)')
axes[1].set_ylabel('Calificación Global')

plt.tight_layout()
plt.show()

print('📈 Estadísticos Descriptivos de Calificaciones:')
display(df[['global_rating', 'global_rating_count']].describe().round(2))


### Top Casas de Perfumería (Diseñadores)


In [ ]:
top_designers = df['designer_raw'].value_counts().head(10).reset_index()
top_designers.columns = ['Diseñador / Casa', 'Cantidad de Fragancias']

plt.figure(figsize=(10, 4.5))
ax = sns.barplot(data=top_designers, x='Cantidad de Fragancias', y='Diseñador / Casa', palette='Blues_r')
plt.title('Top 10 Casas de Perfumería con Mayor Presencia', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Número de Perfumes Registrados')
for p in ax.patches:
    width = p.get_width()
    ax.annotate(f'{int(width)}', (width + 0.1, p.get_y() + p.get_height()/2),
                ha='left', va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 3. Análisis de Pirámides Olfativas (Salida, Corazón y Fondo)


In [ ]:
def extract_notes_frequencies(series):
    notes_list = []
    for entry in series.dropna():
        notes = [n.strip().lower() for n in str(entry).split(',') if n.strip()]
        notes_list.extend(notes)
    return Counter(notes_list)

top_freq = extract_notes_frequencies(df['top_notes_raw'])
heart_freq = extract_notes_frequencies(df['heart_notes_raw'])
base_freq = extract_notes_frequencies(df['base_notes_raw'])

df_top_notes = pd.DataFrame(top_freq.most_common(8), columns=['Nota de Salida', 'Frecuencia'])
df_heart_notes = pd.DataFrame(heart_freq.most_common(8), columns=['Nota de Corazón', 'Frecuencia'])
df_base_notes = pd.DataFrame(base_freq.most_common(8), columns=['Nota de Fondo', 'Frecuencia'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.barplot(data=df_top_notes, x='Frecuencia', y='Nota de Salida', ax=axes[0], palette='Oranges_r')
axes[0].set_title('Notas de Salida (Top Notes)', fontweight='bold', fontsize=11)

sns.barplot(data=df_heart_notes, x='Frecuencia', y='Nota de Corazón', ax=axes[1], palette='YlOrBr_r')
axes[1].set_title('Notas de Corazón (Heart Notes)', fontweight='bold', fontsize=11)

sns.barplot(data=df_base_notes, x='Frecuencia', y='Nota de Fondo', ax=axes[2], palette='Reds_r')
axes[2].set_title('Notas de Fondo (Base Notes)', fontweight='bold', fontsize=11)

plt.suptitle('Ingredientes Más Frecuentes por Nivel de la Pirámide Olfativa', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
## 4. Conclusiones y Guía para la Ingeniería de Variables

### Recomendaciones concretas de preprocesamiento:
1. **Estandarización de Notas:** Aplicar tokenización minúscula y eliminación de plurales/sinónimos (ej. *musk* vs *white musk*).
2. **Ponderación de Pirámide Olfativa:** Al construir el corpus TF-IDF, multiplicar la ocurrencia de notas de fondo por **x3** y corazón por **x2**, ya que determinan el secado (*dry down*) real en la piel.
3. **Extracción de Familias Olfativas Dominantes:** Crear columnas categóricas explicativas (*is_sweet, is_woody, is_fresh, is_oriental*) para permitir filtrado rápido en SQL/FastAPI.


amos a construir un pipeline integral que ataca todas las necesidades:Corrección del Error + Tokenización Robusta.Ponderación de Pirámide Olfativa ($3\times$ fondo, $2\times$ corazón, $1\times$ salida).Mapeos Arquetípicos de Personalidad y Estilo (Elegante, Limpio, Liderazgo/Seriedad, Sedactor, Casual).Ingeniería sobre JSONs Sintéticos (Popularidad, Ratio de Aceptación/Controversia, Score de Estación y Hora del Día).Estrategia Óptima para Reviews (Análisis de Sentimiento VADER / DistilBERT + Aspect Extraction).

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# PASO 1: MANEJO SEGURO DE LISTAS Y NOTAS
# ==========================================

def safe_eval_list(val):
    """Convierte strings representando listas (ej. "['lemon', 'rose']") a listas reales de Python."""
    if isinstance(val, list):
        return val
    if pd.isna(val) or not val:
        return []
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            # Si es un string separado por comas
            return [x.strip() for x in val.split(',') if x.strip()]
    return []

def clean_note(note: str) -> str:
    """Limpia caracteres especiales y estandariza minúsculas."""
    note = str(note).lower().strip()
    note = re.sub(r'[^a-z0-9\s_]', '', note)
    return note.replace(' ', '_')  # Unir n-grams comunes (ej: white_musk)

def create_weighted_corpus(row):
    """Genera el texto con ponderación olfativa de forma segura."""
    top = [clean_note(n) for n in safe_eval_list(row.get('top_notes'))]
    mid = [clean_note(n) for n in safe_eval_list(row.get('heart_notes'))]
    base = [clean_note(n) for n in safe_eval_list(row.get('base_notes'))]
    
    # Si todo está vacío pero existe una columna 'notes' general
    if not (top or mid or base) and 'notes' in row and pd.notna(row['notes']):
        general = [clean_note(n) for n in safe_eval_list(row.get('notes'))]
        return " ".join(general)

    # Multiplicador: Fondo (x3), Corazón (x2), Salida (x1)
    corpus = (top * 1) + (mid * 2) + (base * 3)
    return " ".join(corpus) if corpus else "unknown_note"

# Aplicar a df_fixed
df_fixed['notes_corpus_weighted'] = df_fixed.apply(create_weighted_corpus, axis=1)


# ==========================================
# PASO 2: MAPEOS ARQUETÍPICOS (PERSONALIDAD)
# ==========================================

ARCHETYPES = {
    'is_elegant': ['iris', 'leather', 'sandalwood', 'amber', 'violet', 'cashmere_wood', 'rose', 'vetiver'],
    'is_clean': ['musk', 'white_musk', 'lavender', 'aldehyde', 'neroli', 'soapy_notes', 'cotton', 'bergamot'],
    'is_leadership_boss': ['tobacco', 'oud', 'cedar', 'leather', 'vetiver', 'incense', 'oakmoss', 'spices'],
    'is_seductive': ['vanilla', 'tonka_bean', 'amber', 'cinnamon', 'praline', 'chocolate', 'tuberose', 'rum'],
    'is_fresh_casual': ['lemon', 'citrus', 'mint', 'aquatic_notes', 'marine_notes', 'apple', 'green_notes']
}

for archetype, keywords in ARCHETYPES.items():
    df_fixed[archetype] = df_fixed['notes_corpus_weighted'].apply(
        lambda corpus: int(any(kw in corpus for kw in keywords))
    )


# ==========================================
# PASO 3: INGENIERÍA DE DATOS SINTÉTICOS
# ==========================================

def parse_count(val) -> float:
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        val = val.lower().replace('"', '').strip()
        if 'k' in val:
            return float(val.replace('k', '')) * 1000
        try:
            return float(val)
        except ValueError:
            return 0.0
    return 0.0

def process_synthetic_metrics(row):
    # Parse JSONs
    try:
        vibes = json.loads(row['vibe_reactions_raw_dist']) if isinstance(row['vibe_reactions_raw_dist'], str) else {}
        seasons = json.loads(row['seasons_raw_dist']) if isinstance(row['seasons_raw_dist'], str) else {}
        tod = json.loads(row['time_of_day_raw_dist']) if isinstance(row['time_of_day_raw_dist'], str) else {}
    except Exception:
        vibes, seasons, tod = {}, {}, {}

    # 1. Total Engagement y Popularidad
    love = parse_count(vibes.get('love', 0))
    like = parse_count(vibes.get('like', 0))
    ok = parse_count(vibes.get('ok', 0))
    dislike = parse_count(vibes.get('dislike', 0))
    hate = parse_count(vibes.get('hate', 0))
    
    total_votes = love + like + ok + dislike + hate
    positives = love + like
    negatives = dislike + hate
    
    # Métricas clave
    satisfaction_rate = (positives / total_votes) if total_votes > 0 else 0.5
    controversy_index = (negatives / total_votes) if total_votes > 0 else 0.0
    
    # 2. Season Dominante
    parsed_seasons = {k: parse_count(v) for k, v in seasons.items()}
    best_season = max(parsed_seasons, key=parsed_seasons.get) if parsed_seasons else 'versatile'
    
    # 3. Day / Night Ratio (1.0 = Día puro, 0.0 = Noche pura)
    day = parse_count(tod.get('day', 0))
    night = parse_count(tod.get('night', 0))
    day_ratio = day / (day + night) if (day + night) > 0 else 0.5

    return pd.Series({
        'total_votes': total_votes,
        'satisfaction_rate': satisfaction_rate,
        'controversy_index': controversy_index,
        'best_season': best_season,
        'day_ratio': day_ratio
    })

synthetic_feats = df_fixed.apply(process_synthetic_metrics, axis=1)
df_fixed = pd.concat([df_fixed, synthetic_feats], axis=1)


# ==========================================
# PASO 4: VECTORIZACIÓN Y TF-IDF
# ==========================================

tfidf = TfidfVectorizer(min_df=1, token_pattern=r'(?u)\b\w+\b')
tfidf_matrix = tfidf.fit_transform(df_fixed['notes_corpus_weighted'])

Esta función combina el vector de notas (TF-IDF), la simetría de arquetipos y el filtrado por estación o momento del día.

In [ ]:
def recommend_perfumes(
    perfume_id: int, 
    df: pd.DataFrame, 
    tfidf_mat, 
    top_n: int = 5,
    filter_season: str = None,
    must_have_archetype: str = None,
    min_satisfaction: float = 0.60
):
    """
    Sistema de recomendación híbrido basado en contenido con filtros de negocio.
    """
    # 1. Mapeo de índice
    idx = df.index[df['id'] == perfume_id].tolist()
    if not idx:
        raise ValueError(f"El ID {perfume_id} no se encuentra en el dataset.")
    idx = idx[0]

    # 2. Cosine Similarity Olfativo
    cosine_sim = cosine_similarity(tfidf_mat[idx], tfidf_mat).flatten()
    
    # Crear copie de trabajo con scores
    scores_df = df.copy()
    scores_df['similarity_score'] = cosine_sim

    # Excluir el mismo perfume
    scores_df = scores_df[scores_df.index != idx]

    # 3. Filtros de Calidad y Negocio (Hard Filters)
    scores_df = scores_df[scores_df['satisfaction_rate'] >= min_satisfaction]

    if filter_season:
        scores_df = scores_df[scores_df['best_season'] == filter_season]

    if must_have_archetype and must_have_archetype in df.columns:
        scores_df = scores_df[scores_df[must_have_archetype] == 1]

    # 4. Ajuste del Score Final (Boost por coincidencias de arquetipo)
    target_archetypes = ['is_elegant', 'is_clean', 'is_leadership_boss', 'is_seductive', 'is_fresh_casual']
    target_vector = df.loc[idx, target_archetypes].values.astype(int)
    
    # Bonus si comparte la misma 'vibración'
    archetype_sim = (scores_df[target_archetypes].values @ target_vector) / len(target_archetypes)
    scores_df['final_score'] = (scores_df['similarity_score'] * 0.70) + (archetype_sim * 0.30)

    # Ordenar y retornar top_n
    recommended = scores_df.sort_values(by='final_score', ascending=False)
    
    cols_to_show = ['id', 'name', 'final_score', 'similarity_score', 'best_season', 'satisfaction_rate']
    present_cols = [c for c in cols_to_show if c in recommended.columns]
    
    return recommended[present_cols].head(top_n)

In [ ]:
# 1. Crear un DataFrame de prueba con datos simulados
data_demo = {
    'id': [1, 2, 3, 4, 5],
    'name': ['Bleu Elegance', 'Vanilla Night', 'Citrus Breeze', 'Oud Executive', 'Clean Cotton'],
    'top_notes': ["['lemon', 'bergamot']", "['vanilla', 'cinnamon']", "['lemon', 'mint']", "['spices', 'incense']", "['aldehyde', 'bergamot']"],
    'heart_notes': ["['iris', 'cedar']", "['tonka_bean', 'amber']", "['aquatic_notes']", "['leather', 'oud']", "['lavender', 'white_musk']"],
    'base_notes': ["['sandalwood', 'vetiver']", "['praline', 'chocolate']", "['cedar']", "['tobacco', 'vetiver']", "['musk', 'soapy_notes']"],
    'vibe_reactions_raw_dist': ['{"love": "10k", "like": "2k", "ok": "500", "dislike": "100", "hate": "50"}'] * 5,
    'seasons_raw_dist': ['{"winter": "1k", "spring": "3k", "summer": "5k", "fall": "2k"}'] * 5,
    'time_of_day_raw_dist': ['{"day": "3k", "night": "1k"}'] * 5
}

df_fixed_demo = pd.DataFrame(data_demo)

# Preprocesar datos usando el pipeline anterior
df_fixed_demo['notes_corpus_weighted'] = df_fixed_demo.apply(create_weighted_corpus, axis=1)

for archetype, keywords in ARCHETYPES.items():
    df_fixed_demo[archetype] = df_fixed_demo['notes_corpus_weighted'].apply(
        lambda corpus: int(any(kw in corpus for kw in keywords))
    )

synthetic_feats = df_fixed_demo.apply(process_synthetic_metrics, axis=1)
df_fixed_demo = pd.concat([df_fixed_demo, synthetic_feats], axis=1)

tfidf = TfidfVectorizer(min_df=1, token_pattern=r'(?u)\b\w+\b')
tfidf_matrix = tfidf.fit_transform(df_fixed_demo['notes_corpus_weighted'])

# 2. Ejecutar la recomendación
# Caso de uso: El usuario está viendo 'Bleu Elegance' (id=1) y quiere algo similar pero para estilo 'Elegante/Liderazgo'
recomendaciones = recommend_perfumes(
    perfume_id=1,
    df=df_fixed_demo,
    tfidf_mat=tfidf_matrix,
    top_n=3,
    must_have_archetype='is_leadership_boss'
)

print(recomendaciones)

#### PROCESAMIENTO E INTEGRACION DE REVIEWS

Plan de Procesamiento Recomendado:
Aspect-Based Sentiment Analysis (ABSA):
No hagas un análisis de sentimiento global a toda la review. Un usuario puede decir: "Huele increíble (positivo), pero dura 10 minutos (negativo)".

Extrae entidades clave: duración, estela/proyección, cumplidos, precio/calidad.

Calcula el Perfume Performance Index (PPI):
A partir de las menciones procesadas de las reviews, genera 3 columnas sintéticas de alta precisión:

longevity_score: Frecuencia y sentimiento cuando se menciona "duration", "hours", "lasts".

sillage_score: Frecuencia cuando se menciona "projection", "skin scent", "filling the room".

compliment_factor: Ratio de menciones positivas sobre "compliments", "head turner", "sexy".

Inclusión en el Modelo:
Usa estos scores extraídos de las reviews para aplicar Re-Ranking en la función de recomendación: si el usuario pide "perfume elegante para eventos de noche", la función multiplicará el score por sillage_score y longevity_score.

In [ ]:
import pandas as pd
import numpy as np
import re

# ==============================================================================
# 1. LÉXICO AMPLIADO EN ESPAÑOL (Menciones explícitas e implícitas)
# ==============================================================================
ASPECT_LEXICON_ES = {
    'longevity': {
        'keywords': [
            'duracion', 'durabilidad', 'dura', 'durar', 'fijacion', 'fija', 
            'longevidad', 'horas', 'longevo', 'desempeno', 'performance', 'pellejo'
        ],
        'positives': [
            'excelente', 'bestia', 'barbaridad', 'eterna', 'mucho', 'increible', 
            'buena', '10/10', 'mucha', 'eterno', 'duradero', 'todo el dia', 'sobrado'
        ],
        'negatives': [
            'nada', 'poco', 'debil', 'pobre', 'mala', 'desaparece', 'suspiro', 
            'escasa', 'cero', 'agua', 'mediocre', 'efimero', 'no se siente'
        ]
    },
    'sillage': {
        'keywords': [
            'estela', 'proyeccion', 'proyecta', 'alcance', 'proyectar', 'presencia',
            'se siente', 'distancia', 'estelar'
        ],
        'positives': [
            'enorme', 'pesada', 'potente', 'llena', 'habitacion', 'mucha', 'fuerte', 
            'monstruosa', 'bestial', 'marcada', 'pesado', 'notable'
        ],
        'negatives': [
            'pobre', 'intima', 'baja', 'cerca', 'piel', 'suave', 'nula', 'inexistente',
            'ras de piel', 'timida', 'debil'
        ]
    },
    'compliments': {
        'keywords': [
            'cumplido', 'cumplidos', 'halago', 'halagos', 'elogio', 'elogios', 
            'chulearon', 'preguntaron', 'preguntan', 'gusta a todos', 'encanta a', 
            'llamo la atencion', 'me dijeron', 'te dicen', 'atractivo', 'sexy', 'reacciones'
        ],
        'positives': [
            'muchos', 'siempre', 'reacciones', 'asegurados', 'chulean', 'voltear', 
            'cabezas', 'llueven', 'garantizados', 'increible', 'encanta', 'fascinados'
        ],
        'negatives': [
            'ninguno', 'nadie', 'cero', 'sin', 'ningun', 'feo', 'disgusto', 'desagrada'
        ]
    }
}

# Compilación de patrones Regex
COMPILED_PATTERNS = {}
for aspect, lexicon in ASPECT_LEXICON_ES.items():
    COMPILED_PATTERNS[aspect] = {
        'kw': re.compile(r'\b(' + '|'.join(lexicon['keywords']) + r')\b', re.IGNORECASE),
        'pos': re.compile(r'\b(' + '|'.join(lexicon['positives']) + r')\b', re.IGNORECASE),
        'neg': re.compile(r'\b(' + '|'.join(lexicon['negatives']) + r')\b', re.IGNORECASE)
    }


# ==============================================================================
# 2. FUNCIÓN DE EXTRACCIÓN DE ASPECTOS POR REVIEW
# ==============================================================================
def process_reviews_expanded(df: pd.DataFrame, text_column: str = 'review_text') -> pd.DataFrame:
    """Extrae scores por reseña usando asignación inteligente de NaNs."""
    # Limpieza de acentos vectorizada
    cleaned_series = (
        df[text_column]
        .astype(str)
        .str.lower()
        .str.replace(r'[áàäâ]', 'a', regex=True)
        .str.replace(r'[éèëê]', 'e', regex=True)
        .str.replace(r'[íìïî]', 'i', regex=True)
        .str.replace(r'[óòöô]', 'o', regex=True)
        .str.replace(r'[úùüû]', 'u', regex=True)
    )

    results = pd.DataFrame(index=df.index)

    for aspect, patterns in COMPILED_PATTERNS.items():
        has_aspect = cleaned_series.str.contains(patterns['kw'], regex=True)

        pos_counts = cleaned_series.str.findall(patterns['pos']).str.len()
        neg_counts = cleaned_series.str.findall(patterns['neg']).str.len()
        total_modifiers = pos_counts + neg_counts

        # Cálculo de Score:
        # Si NO menciona el aspecto -> np.nan (No ensucia el promedio con ceros)
        # Si menciona el aspecto pero no hay modificadores claros -> 0.1 (Neutro leve)
        # Si hay modificadores -> (Pos - Neg) / Total
        score = np.where(
            ~has_aspect,
            np.nan,
            np.where(
                total_modifiers > 0,
                (pos_counts - neg_counts) / total_modifiers,
                0.1
            )
        )

        results[f'{aspect}_score'] = np.round(score, 2)

    return results


# ==============================================================================
# 3. AGREGACIÓN BAYESIANA PONDERADA POR PERFUME
# ==============================================================================
def calculate_bayesian_perfume_scores(reviews_df: pd.DataFrame, aspect_scores_df: pd.DataFrame, min_mencions_weight: int = 3) -> pd.DataFrame:
    """
    Agrupa por perfume y calcula un Promedio Bayesiano para suavizar la distribución.
    """
    df_combined = pd.concat([reviews_df[['perfume_id']], aspect_scores_df], axis=1)

    perfume_metrics = {}
    
    for aspect in ['longevity_score', 'sillage_score', 'compliments_score']:
        # Media global del aspecto (excluyendo NaNs)
        global_mean = df_combined[aspect].mean()
        if pd.isna(global_mean):
            global_mean = 0.0

        # Agregación por perfume (Cuenta menciones reales y calcula el promedio crudo)
        grouped = df_combined.groupby('perfume_id')[aspect].agg(['count', 'mean']).reset_index()

        # Fórmula Bayesiana: (n * mean + m * global_mean) / (n + m)
        # n = número de reviews que mencionan el aspecto
        # m = peso mínimo de suavizado (m menciones virtuales)
        n = grouped['count']
        mean = grouped['mean'].fillna(global_mean)
        m = min_mencions_weight

        bayesian_score = (n * mean + m * global_mean) / (n + m)
        grouped[f'{aspect}_bayesian'] = np.round(bayesian_score, 3)

        perfume_metrics[aspect] = grouped[['perfume_id', f'{aspect}_bayesian']]

    # Unir los 3 aspectos en un único DataFrame final por perfume
    final_perfume_df = perfume_metrics['longevity_score']
    final_perfume_df = final_perfume_df.merge(perfume_metrics['sillage_score'], on='perfume_id')
    final_perfume_df = final_perfume_df.merge(perfume_metrics['compliments_score'], on='perfume_id')

    return final_perfume_df


# ==============================================================================
# 4. EJECUCIÓN DEL PIPELINE COMPLETO
# ==============================================================================

reviews_demo = df_fixed[['perfume_id', 'reviews_text_corpus']].copy()

# 1. Procesar reviews individuales
aspect_scores = process_reviews_expanded(reviews_demo, text_column='reviews_text_corpus')

# 2. Calcular los scores finales suavizados para los perfumes
perfume_performance_df = calculate_bayesian_perfume_scores(reviews_demo, aspect_scores)

print("--- METRICAS CORREGIDAS Y SUAVIZADAS (DISTRIBUCIÓN CONTINUA) ---")
print(perfume_performance_df)

In [ ]:
## estadisticos de performance_df
perfomance_stats = perfume_performance_df[['longevity_score_bayesian', 'sillage_score_bayesian', 'compliments_score_bayesian']].describe().round(3)
print("\n--- ESTADÍSTICOS DESCRIPTIVOS DE METRICAS ---")
print(perfomance_stats)

In [ ]:
# Valores unicos de las metricas de perfume_performance_df
unique_longevity = perfume_performance_df['longevity_score_bayesian'].nunique()
unique_sillage = perfume_performance_df['sillage_score_bayesian'].nunique()
unique_compliment = perfume_performance_df['compliments_score_bayesian'].nunique()
print(f"\nNúmero de valores únicos para cada métrica:")
print(f"Longevidad: {unique_longevity}, Estela: {unique_sillage}, Halagos: {unique_compliment}")

print("Valores unicos de cada métrica:")
print(f"Longevidad: {perfume_performance_df['longevity_score_bayesian'].unique()}")
print(f"Estela: {perfume_performance_df['sillage_score_bayesian'].unique()}")
print(f"Halagos: {perfume_performance_df['compliments_score_bayesian'].unique()}")

#Distribución de métricas derivadas
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(perfume_performance_df['longevity_score_bayesian'], kde=True)
plt.title('Distribución de Longevidad')

plt.subplot(1, 3, 2)
sns.histplot(perfume_performance_df['sillage_score_bayesian'], kde=True)
plt.title('Distribución de Estela')

plt.subplot(1, 3, 3)
sns.histplot(perfume_performance_df['compliments_score_bayesian'], kde=True)
plt.title('Distribución de Halagos')

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
perfume_performance_df[['longevity_norm', 'sillage_norm', 'compliments_norm']] = scaler.fit_transform(
    perfume_performance_df[['longevity_score_bayesian', 'sillage_score_bayesian', 'compliments_score_bayesian']]
)

# Nuevas distribuciones normalizadas
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(perfume_performance_df['longevity_norm'], kde=True)
plt.title('Distribución de Longevidad Normalizada')

plt.subplot(1, 3, 2)
sns.histplot(perfume_performance_df['sillage_norm'], kde=True)
plt.title('Distribución de Estela Normalizada')

plt.subplot(1, 3, 3)
sns.histplot(perfume_performance_df['compliments_norm'], kde=True)
plt.title('Distribución de Halagos Normalizada')

plt.tight_layout()
plt.show()


podemos integrar las variables construidas (corpus TF-IDF ponderado, arquetipos, métricas bayesianas de reviews y distribución de sensaciones) para resolver dos tareas centrales:

Aprendizaje No Supervisado (Clustering / Profiling): Descubrir de forma automática los Perfiles Olfativos Subyacentes del mercado (agrupando perfumes según notas, estela, duración y arquetipos).

Sistema de Recomendación Híbrido (Content-Based + ML Vector Embeddings): Generar un espacio vectorial donde cada perfume y cada perfil de usuario tengan una posición matemática clara para recomendar por distancia/similitud.

''' text
[ Variables Generadas ]
     (TF-IDF Ponderado + Arquetipos + Metricas Reviews + Sintéticos)
                                 │
                                 ▼
                     [ Reducción de Dimensionalidad ]
                       (TruncatedSVD / PCA / UMAP)
                                 │
                                 ▼
                   [ Modelado No Supervisado (Clustering) ]
                   (K-Means / HDBSCAN / Gaussian Mixtures)
                                 │
                                 ▼
            ┌────────────────────┴────────────────────┐
            ▼                                         ▼
   [ Perfilamiento Olfativo ]               [ Motor de Recomendación ]
 - Segmentación automática de             - Mapeo de Usuario -> Perfil
   familias / arquetipos.                 - Cosine Similarity en Espacio
 - Caracterización por clúster.             Latente Reducido.


## INTEGRACION DE CARACTERISTICAS

In [ ]:
import pandas as pd
import numpy as np
import ast
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

# ==============================================================================
# 1. PARSEO ROBUSO DE NOTAS USANDO LAS COLUMNAS _raw REALES
# ==============================================================================

def parse_any_notes_to_list(val) -> list:
    """Convierte cadenas/listas de notas en una lista limpia de strings."""
    if isinstance(val, list):
        return [str(x).strip() for x in val if x]
    if pd.isna(val) or not val or val == 'nan':
        return []
    if isinstance(val, str):
        val_str = val.strip()
        if val_str.startswith('[') and val_str.endswith(']'):
            try:
                parsed = ast.literal_eval(val_str)
                if isinstance(parsed, list):
                    return [str(x).strip() for x in parsed if x]
            except Exception:
                pass
        cleaned = re.sub(r"[\[\]'\"']", "", val_str)
        return [x.strip() for x in re.split(r'[,|/]', cleaned) if x.strip()]
    return []

def clean_note_text(note: str) -> str:
    note = str(note).lower().strip()
    note = re.sub(r'[^a-z0-9\s_]', '', note)
    return note.replace(' ', '_')

def create_weighted_corpus_from_raw(row):
    """Ponderación olfativa 3x Fondo, 2x Corazón, 1x Salida sobre columnas _raw."""
    top = [clean_note_text(n) for n in parse_any_notes_to_list(row.get('top_notes_raw'))]
    mid = [clean_note_text(n) for n in parse_any_notes_to_list(row.get('heart_notes_raw'))]
    base = [clean_note_text(n) for n in parse_any_notes_to_list(row.get('base_notes_raw'))]
    
    corpus = (top * 1) + (mid * 2) + (base * 3)
    return " ".join(corpus) if corpus else "unknown_note"

df_master = df_fixed.merge(perfume_performance_df, on='perfume_id', how='left')


# Re-generar el corpus con las columnas correctas
df_master['notes_corpus_weighted'] = df_master.apply(create_weighted_corpus_from_raw, axis=1)

print("--- REVISIÓN DE CORPUS GENERADO ---")
print(f"Total de registros con 'unknown_note': {(df_master['notes_corpus_weighted'] == 'unknown_note').sum()}")
print("Muestra del corpus:", df_master['notes_corpus_weighted'].head(2).tolist())


# ==============================================================================
# 2. VECTORIZACIÓN TF-IDF & TRUNCATED SVD
# ==============================================================================

tfidf = TfidfVectorizer(min_df=1, token_pattern=r'(?u)\b\w+\b')
tfidf_matrix = tfidf.fit_transform(df_master['notes_corpus_weighted'])

# Columnas numéricas para el espacio latente
num_cols = [
    'longevity_norm', 
    'sillage_norm', 
    'compliments_norm',
    'day_ratio', 
    'satisfaction_rate', 
    'controversy_index',
    'is_elegant', 
    'is_clean', 
    'is_leadership_boss', 
    'is_seductive', 
    'is_fresh_casual'
]

# Imputar nulos en numéricas si existieran
df_master[num_cols] = df_master[num_cols].fillna(0)

scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(df_master[num_cols])

# Concatenar matrices
X_combined = hstack([tfidf_matrix, X_numeric_scaled]).tocsr()

n_samples, n_features = X_combined.shape
n_comp = min(15, min(n_samples, n_features) - 1)

svd = TruncatedSVD(n_components=n_comp, random_state=42)
X_embedding = svd.fit_transform(X_combined)


# ==============================================================================
# 3. ENTRENAMIENTO DE K-MEANS (9 CLÚSTERES)
# ==============================================================================

kmeans_model = KMeans(n_clusters=9, random_state=42, n_init=10)
df_master['olfactory_cluster'] = kmeans_model.fit_predict(X_embedding)
df_master['distance_to_cluster_center'] = kmeans_model.transform(X_embedding).min(axis=1)


# ==============================================================================
# 4. GENERACIÓN DEL PERFILAMIENTO FINAL DE CLÚSTERES
# ==============================================================================

def generate_cluster_profiles(df, tfidf_vectorizer, tfidf_mat, num_columns, cluster_col='olfactory_cluster'):
    feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
    clusters = sorted(df[cluster_col].unique())
    global_means = df[num_columns].mean()
    
    profiles = []

    for c in clusters:
        cluster_mask = (df[cluster_col] == c)
        cluster_df = df[cluster_mask]
        cluster_size = len(cluster_df)
        
        # 1. Variables numéricas destacadas
        cluster_means = cluster_df[num_columns].mean()
        diff = cluster_means - global_means
        
        top_positive_traits = diff.sort_values(ascending=False).head(3).to_dict()
        top_negative_traits = diff.sort_values(ascending=True).head(2).to_dict()

        # 2. Top 5 notas TF-IDF reales del clúster
        cluster_tfidf = tfidf_mat[cluster_mask.values].mean(axis=0)
        top_note_indices = np.argsort(np.asarray(cluster_tfidf).flatten())[::-1][:5]
        top_notes = feature_names[top_note_indices].tolist()

        profiles.append({
            'cluster_id': c,
            'size': cluster_size,
            'pct_of_total': f"{(cluster_size / len(df)) * 100:.1f}%",
            'top_notes': ", ".join(top_notes),
            'high_traits': [f"{k} (+{v:.2f})" for k, v in top_positive_traits.items() if v > 0.02],
            'low_traits': [f"{k} ({v:.2f})" for k, v in top_negative_traits.items() if v < -0.02]
        })

    return pd.DataFrame(profiles)

df_profiles = generate_cluster_profiles(df_master, tfidf, tfidf_matrix, num_cols)

# Mostrar la tabla de perfiles limpia
pd.set_option('display.max_colwidth', None)
print("\n--- PERFILAMIENTO FINAL DE LOS 9 CLÚSTERES OLFATIVOS ---")
print(df_profiles[['cluster_id', 'size', 'pct_of_total', 'top_notes', 'high_traits','low_traits']])

Mapeo de Etiquetas por Clúster
Clúster 0: Baja Fijación / Bruma Casual o Agua Colonia
Atributos Clave: longevity_norm (-0.45), sillage_norm (-0.16).

Interpretación: Fragancias con desempeño sumamente sutil o de muy corta duración en piel. Ideales para uso íntimo, relajación o aplicación rápida tras la ducha.

Etiqueta Sugerida: Bruma Casual & Sombra Intima

Clúster 1: Elegancia Nocturna Masiva
Atributos Clave: day_ratio (-0.28), size: 21.0% (uno de los grupos más grandes).

Interpretación: Es un perfil muy poblado orientado marcadamente al uso nocturno, de baja controversia y alta aceptación general.

Etiqueta Sugerida: Nocturno Urbano & Aceptación Versátil

Clúster 2: Nicho Polarizante & Amaderado Intenso
Atributos Clave: controversy_index (+0.26), satisfaction_rate (-0.16).

Etiqueta Sugerida: Nicho Polarizante & Amaderado Audaz

Clúster 3: Mass-Pleaser / Superventas Seguro
Atributos Clave: satisfaction_rate (+0.19), controversy_index (-0.16), size: 20.7%.

Interpretación: Puntuación de satisfacción muy alta y bajísima controversia. Son las fragancias "a ciegas" seguras que le gustan a prácticamente todos.

Etiqueta Sugerida: Mass-Pleaser & Apuesta Segura

Clúster 4: Cítrico / Fresco Diurno de Oficina
Atributos Clave: day_ratio (+0.27), size: 20.4%.

Interpretación: Fragancias dominadas por un uso predominantemente solar y de día.

Etiqueta Sugerida: Fresco Diurno & Estilo Oficina

Clúster 5: El Imán de Cumplidos (Compliment Getter)
Atributos Clave: compliments_norm (+0.47), longevity_norm (+0.05).

Interpretación: Destaca de forma sobresaliente en la métrica de halagos y reacciones positivas de terceros según las reviews.

Etiqueta Sugerida: Seductor & Imán de Cumplidos

Clúster 6: Gourmand Dulce Intimo
Atributos Clave: Presencia clara de haba_tonka y mbar, compliments_norm (-0.39).

Interpretación: Acordes dulces y cálidos de disfrute personal que no buscan proyectar ni gustar masivamente a los demás.

Etiqueta Sugerida: Gourmand Cálido & Calidez Personal

Clúster 7: Proyección Monstruosa / Modo Bestia (Sillage King)
Atributos Clave: sillage_norm (+0.57) (la proyección más alta de todo el dataset).

Interpretación: Fragancias que llenan habitaciones y dejan una huella imborrable a su paso.

Etiqueta Sugerida: Estela Potente & Presencia Impuesta

Clúster 8: Eterno en Piel (Longevity Beast)
Atributos Clave: longevity_norm (+0.43).

Interpretación: Destaca por fijarse de forma extraordinaria a la piel durante jornadas muy extensas.

Etiqueta Sugerida: Larga Duración & Alta Fijación

In [ ]:
# Diccionario de asignación de nombres de perfil
olfactory_profile_map = {
    0: "Bruma Casual & Sombra Íntima",
    1: "Nocturno Urbano & Aceptación Versátil",
    2: "Nicho Polarizante & Amaderado Audaz",
    3: "Mass-Pleaser & Apuesta Segura",
    4: "Fresco Diurno & Estilo Oficina",
    5: "Seductor & Imán de Cumplidos",
    6: "Gourmand Cálido & Calidez Personal",
    7: "Estela Potente & Presencia Impuesta",
    8: "Larga Duración & Alta Fijación"
}

# Asignar la columna explicativa en df_master
df_master['olfactory_profile_label'] = df_master['olfactory_cluster'].map(olfactory_profile_map)

# Verificar distribución final por nombre de perfil
print(df_master['olfactory_profile_label'].value_counts())

Recomendación Basada en Perfil Olfativo del Usuario
Imagina que un nuevo usuario completa un test o expresa su preferencia (ej. "busco algo limpio, elegante y de buena duración"). Podemos vectorizar el perfil del usuario, proyectarlo en el mismo espacio vectorial de Machine Learning y calcular los perfumes más cercanos dentro de su mismo clúster olfativo.

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics.pairwise import cosine_similarity

# Diccionario de asignación de nombres de perfil
olfactory_profile_map = {
    0: "Bruma Casual & Sombra Íntima",
    1: "Nocturno Urbano & Aceptación Versátil",
    2: "Nicho Polarizante & Amaderado Audaz",
    3: "Mass-Pleaser & Apuesta Segura",
    4: "Fresco Diurno & Estilo Oficina",
    5: "Seductor & Imán de Cumplidos",
    6: "Gourmand Cálido & Calidez Personal",
    7: "Estela Potente & Presencia Impuesta",
    8: "Larga Duración & Alta Fijación"
}


def recommend_by_user_preference(
    user_preferences: dict, 
    df: pd.DataFrame, 
    X_embed, 
    tfidf_mat,
    model_kmeans, 
    scaler_obj, 
    svd_obj, 
    top_n: int = 5
):
    """
    Recomienda perfumes proyectando las preferencias del usuario al espacio vectorial del modelo.
    """
    # 1. Construir el vector numérico del usuario respetando el orden exacto de num_cols
    user_num_vector = np.array([[
        user_preferences.get('longevity_norm', 0.5),
        user_preferences.get('sillage_norm', 0.5),
        user_preferences.get('compliments_norm', 0.5),
        user_preferences.get('day_ratio', 0.5),
        user_preferences.get('satisfaction_rate', 0.8),
        user_preferences.get('controversy_index', 0.1),
        user_preferences.get('is_elegant', 0),
        user_preferences.get('is_clean', 0),
        user_preferences.get('is_leadership_boss', 0),
        user_preferences.get('is_seductive', 0),
        user_preferences.get('is_fresh_casual', 0)
    ]])

    # Escalado con el StandardScaler entrenado
    user_num_scaled = scaler_obj.transform(user_num_vector)
    
    # Matriz sparse de ceros con la forma EXACTA de 1 fila x N columnas de TF-IDF
    n_tfidf_features = tfidf_mat.shape[1]
    user_tfidf_dummy = csr_matrix((1, n_tfidf_features), dtype=np.float64)
    
    # Concatenar vector de texto (ceros) con vector numérico escalado
    user_combined = hstack([user_tfidf_dummy, user_num_scaled]).tocsr()
    
    # Proyección al espacio reducido TruncatedSVD
    user_embedding = svd_obj.transform(user_combined)

    # 2. Predecir el clúster olfativo
    predicted_cluster = model_kmeans.predict(user_embedding)[0]
    predicted_profile = olfactory_profile_map.get(predicted_cluster, f"Clúster {predicted_cluster}")

    # 3. Filtrar perfumes que pertenecen al mismo perfil
    cluster_mask = (df['olfactory_cluster'] == predicted_cluster)
    cluster_indices = np.where(cluster_mask)[0]
    
    if len(cluster_indices) == 0:
        cluster_indices = np.arange(len(df))

    # 4. Similitud del coseno en el espacio latente
    sims = cosine_similarity(user_embedding, X_embed[cluster_indices]).flatten()
    
    # 5. Formatear y ordenar los resultados
    results = df.iloc[cluster_indices].copy()
    results['similarity'] = sims
    
    recommended = results.sort_values(by='similarity', ascending=False).head(top_n)

    # Seleccionar columnas disponibles para mostrar
    cols_to_show = ['name_raw', 'designer_raw', 'olfactory_profile_label', 'similarity', 'best_season', 'satisfaction_rate']
    present_cols = [c for c in cols_to_show if c in recommended.columns]

    return predicted_profile, recommended[present_cols]


# ==========================================
# EJEMPLO DE EJECUCIÓN
# ==========================================

user_test = {
    'is_clean': 1,
    'is_fresh_casual': 1,
    'day_ratio': 0.85,
    'longevity_norm': 0.6,
    'sillage_norm': 0.4
}

profile, recs = recommend_by_user_preference(
    user_preferences=user_test,
    df=df_master,
    X_embed=X_embedding,
    tfidf_mat=tfidf_matrix,
    model_kmeans=kmeans_model,
    scaler_obj=scaler,
    svd_obj=svd,
    top_n=5
)

print(f"\n--- PERFIL OLFATIVO PREDICHO PARA EL USUARIO: {profile} ---\n")
print(recs.to_string(index=False))